# 01b - Exploration of the GCM runs: EC-Earth3 historical and the hosing experiments

**Purpose**: know the GCM data before any correction, with a focus on what the bias adjustment must **preserve**:
the climate-change signal and the **hosing signal** (hosing - original).

Runs: EC-Earth3 historical (CMIP6, r1i1p1f1) and the four stabilization runs (2025 / 2050, original / hosing).
Variables: `pr` (mm/day), `tasmax`, `tasmin` (degC) and the diurnal range `dtr = tasmax - tasmin`.

**Domain**: maps, means, zonal profiles and distributions are computed over the **LISFLOOD domain** (land incl. lakes,
no Antarctica), i.e. the mask of notebook 00c transferred to the native EC-Earth3 grid with the rule
"a GCM pixel is inside if at least one of its ERA5 pixels is inside". Set `USE_DOMAIN_MASK = False` for the whole globe.

**How it works**: each run is read **once** (~10 GB per variable) and reduced to compact statistics
(yearly means, monthly climatology, yearly extremes, dry days, pooled quantiles) stored in
`<out_root>/exploration_gcm/cache/`. All figures are made from this cache: the first run of section 0 is long
(reading ~30 GB per run), then everything is fast. Set `OVERWRITE_CACHE = True` to recompute.

Sections: 1. integrity - 2. stationarity - 3. historical climatology - 4. signal maps - 5. zonal means -
6. annual cycle - 7. extremes - 8. dry days and wet-day intensity - 9. distributions - 10. interannual variability.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import dask
from dask.diagnostics import ProgressBar
from scipy import stats
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']

RUNS = {"gcm_historical": "historical",
        "gcm_2025_nohosing": "2025 original", "gcm_2025_hosing": "2025 hosing",
        "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing"}
COLORS = {"gcm_historical": "k", "gcm_2025_nohosing": "tab:blue", "gcm_2025_hosing": "tab:red",
          "gcm_2050_nohosing": "tab:cyan", "gcm_2050_hosing": "tab:orange"}
UNITS = {"pr": "mm/day", "tasmax": "degC", "tasmin": "degC", "dtr": "degC"}
N_SAMPLE = 3000          # area-weighted random pixels used for the pooled distributions (section 9)
QUANTILES = [0.001, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
OVERWRITE_CACHE = False
USE_DOMAIN_MASK = True   # True = LISFLOOD domain (notebook 00c must have been run), False = whole globe
OUT = bu.out_dir(cfg, "exploration_gcm")
CACHE = bu.out_dir(cfg, "exploration_gcm", "cache")

try:                                   # coastlines if cartopy is installed, plain maps otherwise
    import cartopy.crs as ccrs
    PROJ = ccrs.PlateCarree()
except ImportError:
    PROJ = None
print("maps with coastlines:", PROJ is not None)

# ---- domain mask on the native EC-Earth3 grid (all the GCM runs share this grid, notebook 00b)
_ref = next(n for n in RUNS if all(bu.dataset_available(cfg, n, v) for v in VARS))
GRID = bu.read_raw(cfg, _ref, "tasmax").isel(time=0)
if USE_DOMAIN_MASK:
    assert bu.domain_mask_path(cfg).exists(), "Domain mask not found: run notebook 00c_land_mask first."
    MASK = bu.load_domain_mask(cfg, like=GRID)
else:
    MASK = xr.ones_like(GRID, dtype=bool)
DOMAIN = "LISFLOOD domain" if USE_DOMAIN_MASK else "globe"
print(f"{DOMAIN}: {int(MASK.sum())} of {MASK.size} GCM pixels")

## 0. Statistics of each run (computed once, then read from the cache)

Per run, in a single pass over the data:
- `<v>_clim_month` : monthly climatology (month, lat, lon)
- `<v>_year_mean` : yearly means (year, lat, lon)
- `<v>_year_ext` : yearly maximum (`pr` = Rx1day, `tasmax` = TXx) or minimum (`tasmin` = TNn)
- `pr_year_dry`, `pr_month_dry` : dry-day fraction (pr < 0.1 mm/day); `pr_year_sdii` : mean wet-day intensity
- `<v>_q` : pooled quantiles on `N_SAMPLE` random pixels of the domain (area-weighted), `pr_q_wet` : wet days only
- integrity scalars (NaN fraction, pr < 0, tasmin > tasmax)

In [ ]:
def sample_points(lat, lon, n, seed=0):
    """Random pixels of the domain, probability proportional to their area (cos lat), same pixels for every run."""
    rng = np.random.default_rng(seed)
    w = (np.repeat(np.cos(np.deg2rad(lat))[:, None], lon.size, axis=1) * MASK.values).ravel()
    idx = rng.choice(w.size, size=n, replace=False, p=w / w.sum())
    return np.unravel_index(idx, (lat.size, lon.size))


def run_stats(name):
    path = CACHE / f"{name}_stats_{'domain' if USE_DOMAIN_MASK else 'globe'}.nc"
    if path.exists() and not OVERWRITE_CACHE:
        return xr.open_dataset(path).load()
    print(f"-> computing {name} (reads the whole run once)")
    d = {v: bu.read_raw(cfg, name, v) for v in VARS}
    d["dtr"] = d["tasmax"] - d["tasmin"]
    iy, ix = sample_points(d["pr"].lat.values, d["pr"].lon.values, N_SAMPLE)
    pick = dict(lat=xr.DataArray(iy, dims="point"), lon=xr.DataArray(ix, dims="point"))
    out, pts = {}, {}
    for v, da in d.items():
        out[f"{v}_clim_month"] = da.groupby("time.month").mean()
        out[f"{v}_year_mean"] = da.groupby("time.year").mean()
        pts[v] = da.isel(**pick)
    out["pr_year_ext"] = d["pr"].groupby("time.year").max()
    out["tasmax_year_ext"] = d["tasmax"].groupby("time.year").max()
    out["tasmin_year_ext"] = d["tasmin"].groupby("time.year").min()
    dry = (d["pr"] < bu.THRESH_PR).where(d["pr"].notnull())
    out["pr_year_dry"] = dry.groupby("time.year").mean()
    out["pr_month_dry"] = dry.groupby("time.month").mean()
    out["pr_year_sdii"] = d["pr"].where(d["pr"] >= bu.THRESH_PR).groupby("time.year").mean()
    scal = {f"nan_frac_{v}": d[v].isnull().mean() for v in VARS}
    scal["frac_pr_negative"] = (d["pr"] < 0).mean()
    scal["frac_pr_below_m0.01"] = (d["pr"] < -0.01).mean()
    scal["pr_min"] = d["pr"].min()
    scal["frac_tasmin_gt_tasmax"] = (d["tasmin"] > d["tasmax"]).mean()
    with ProgressBar():
        out, pts, scal = dask.compute(out, pts, scal)
    ds = xr.Dataset({k: v.astype("float32") for k, v in out.items()})
    ds = ds.assign_coords(q=QUANTILES)
    for v, p in pts.items():
        a = p.values.ravel(); a = a[np.isfinite(a)]
        ds[f"{v}_q"] = ("q", np.quantile(a, QUANTILES).astype("float32"))
        if v == "pr":
            w = a[a >= bu.THRESH_PR]
            ds["pr_q_wet"] = ("q", np.quantile(w, QUANTILES).astype("float32"))
            ds.attrs["pooled_dry_frac"] = float((a < bu.THRESH_PR).mean())
    for k, val in scal.items():
        ds.attrs[k] = float(val)
    ds.attrs["run"] = name
    ds.to_netcdf(path)
    return ds


S = {}
for name in RUNS:
    if all(bu.dataset_available(cfg, name, v) for v in VARS):
        S[name] = run_stats(name)
    else:
        print(f"[missing] {name}: path not filled in or no file (notebook 00)")
print("runs available:", [RUNS[n] for n in S])

In [ ]:
# ---- helpers used by all the figures
def seas(ds, v, season):
    """Climatological map: 'ANN' (mean of the yearly means), 'DJF' or 'JJA' (from the monthly climatology)."""
    if season == "ANN":
        return ds[f"{v}_year_mean"].mean("year")
    months = {"DJF": [12, 1, 2], "JJA": [6, 7, 8], "MAM": [3, 4, 5], "SON": [9, 10, 11]}[season]
    return ds[f"{v}_clim_month"].sel(month=months).mean("month")


def welch_sig(a, b, t_crit=2.0):
    """|Welch t| > t_crit on yearly values (dim 'year'). Serial correlation is ignored: indicative only."""
    ma, mb = a.mean("year"), b.mean("year")
    se = np.sqrt(a.var("year", ddof=1) / a.sizes["year"] + b.var("year", ddof=1) / b.sizes["year"])
    return np.abs(ma - mb) / se > t_crit


def diff(a, b, v):
    """Difference a - b; relative (%) for pr, masked where the reference is below 0.1 mm/day."""
    if v == "pr":
        return (100 * (a - b) / b).where(b > 0.1)
    return a - b


def dmean(da):
    """Area-weighted mean over the domain."""
    return bu.area_mean(da.where(MASK))


def map_axes(nr, nc, w=5.0, h=2.7):
    kw = {"subplot_kw": {"projection": PROJ}} if PROJ is not None else {}
    fig, axes = plt.subplots(nr, nc, figsize=(w * nc, h * nr), squeeze=False, **kw)
    return fig, axes


def draw(ax, da, title, cmap="viridis", lim=None, sym=False, sig=None, label=""):
    da = da.where(MASK)                       # outside the domain: grey
    if sig is not None:
        sig = sig & MASK
    ax.set_facecolor("0.85")
    if sym:
        lim = lim or float(np.nanpercentile(np.abs(da.values), 98)) or 1.0
        vmin, vmax = -lim, lim
    else:
        vmin, vmax = (None, None) if lim is None else lim
    kw = {"transform": PROJ} if PROJ is not None else {}
    im = ax.pcolormesh(da.lon, da.lat, da, shading="auto", cmap=cmap, vmin=vmin, vmax=vmax, **kw)
    if sig is not None:
        ax.contourf(sig.lon, sig.lat, sig.astype(float), levels=[0.5, 1.5], hatches=["...."], colors="none", **kw)
    if PROJ is not None:
        try:
            ax.coastlines(lw=0.4)
        except Exception:
            pass
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8, label=label)


def years(ds):
    y = ds["year"].values
    return f"{y.min()}-{y.max()} ({y.size} y)"

In [ ]:
fig, axes = map_axes(1, 1, w=9, h=4)
draw(axes[0, 0], MASK.astype(float), f"{DOMAIN} on the native EC-Earth3 grid ({int(MASK.sum())} pixels)", lim=(0, 1))
plt.tight_layout(); plt.savefig(OUT / "domain_mask_gcm_grid.png", dpi=110); plt.show()

## 1. Integrity of each run

In [ ]:
rows = []
for name, ds in S.items():
    r = {"run": RUNS[name], "years": f"{int(ds.year.min())}-{int(ds.year.max())}", "n years": ds.sizes["year"]}
    for v in VARS:
        r[f"NaN {v}"] = ds.attrs[f"nan_frac_{v}"]
    r["frac pr < 0"] = ds.attrs["frac_pr_negative"]
    r["pr min (mm/day)"] = ds.attrs["pr_min"]
    r["frac pr < -0.01"] = ds.attrs["frac_pr_below_m0.01"]
    r["frac tasmin > tasmax"] = ds.attrs["frac_tasmin_gt_tasmax"]
    for v in ("pr", "tasmax", "tasmin", "dtr"):
        r[f"{DOMAIN} mean {v}"] = round(float(dmean(seas(ds, v, "ANN"))), 3)
    for v in ("pr", "tasmax", "tasmin", "dtr"):
        r[f"globe mean {v}"] = round(float(bu.area_mean(seas(ds, v, "ANN"))), 3)
    r["pooled dry-day frac"] = round(ds.attrs["pooled_dry_frac"], 3)
    rows.append(r)
integrity = pd.DataFrame(rows).set_index("run")
integrity.to_csv(OUT / "integrity.csv")
with pd.option_context("display.width", 250, "display.max_columns", 30):
    display(integrity)

Note: EC-Earth3 has many *tiny* negative precipitation values (about -1e-19 mm/day, numerical zeros): they are
harmless (below the 0.1 mm/day dry-day threshold). What matters is `pr min` and the fraction below -0.01 mm/day.

Expected: no NaN, no clearly negative precipitation, `tasmin > tasmax` very rare (0 ideally). Anything else must be
understood before the bias adjustment (WDF and QDM assume physically consistent inputs).

## 2. Stationarity: yearly means over the domain and trends

The stabilization runs are meant to be in a (quasi-)stable climate. A trend inside a run matters for the method:
the corrections are calibrated over the whole run (2025 original), and the two-halves validation compares halves of it.
The historical run is expected to warm.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 3.8))
trend_rows = []
for ax, v in zip(axes, ("pr", "tasmax", "tasmin", "dtr")):
    for name, ds in S.items():
        g = dmean(ds[f"{v}_year_mean"])
        ax.plot(g.year, g, color=COLORS[name], lw=1, label=RUNS[name])
        lr = stats.linregress(g.year.values, g.values)
        half = g.sizes["year"] // 2
        trend_rows.append({"run": RUNS[name], "var": v, "trend per decade": round(10 * lr.slope, 4),
                           "p-value": round(lr.pvalue, 4),
                           "2nd half - 1st half": round(float(g[half:].mean() - g[:half].mean()), 4)})
    ax.set_title(f"{DOMAIN} mean {v} ({UNITS[v]})"); ax.set_xlabel("model year")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "yearly_means.png", dpi=110); plt.show()
trends = pd.DataFrame(trend_rows).pivot(index="run", columns="var")
trends.to_csv(OUT / "trends.csv")
display(trends)

## 3. Climatology of the historical run (1981-2014)

In [ ]:
h = S.get("gcm_historical")
if h is not None:
    fig, axes = map_axes(3, 3)
    for r, v in enumerate(VARS):
        for c, s in enumerate(("ANN", "DJF", "JJA")):
            cmap = "YlGnBu" if v == "pr" else "RdYlBu_r"
            lim = (0, float(np.nanpercentile(seas(h, v, s).where(MASK), 98))) if v == "pr" else None
            draw(axes[r, c], seas(h, v, s), f"historical {v} {s}", cmap=cmap, lim=lim, label=UNITS[v])
    plt.tight_layout(); plt.savefig(OUT / "historical_climatology.png", dpi=110); plt.show()

## 4. Signal maps

Differences of climatologies (pr in %). Dots: |Welch t| > 2 on the **yearly** values (annual column only;
serial correlation ignored, so this is indicative). These are the signals that F1 / F2 must not distort:
compare later with the same maps after bias adjustment.

In [ ]:
PAIRS = [("gcm_2025_nohosing", "gcm_historical", "2025 original - historical (climate change)"),
         ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original"),
         ("gcm_2050_nohosing", "gcm_2025_nohosing", "2050 original - 2025 original")]
for a, b, lab in PAIRS:
    if a not in S or b not in S:
        print(f"[missing] {lab}"); continue
    fig, axes = map_axes(3, 3)
    for r, v in enumerate(VARS):
        sig = welch_sig(S[a][f"{v}_year_mean"], S[b][f"{v}_year_mean"])
        for c, s in enumerate(("ANN", "DJF", "JJA")):
            d = diff(seas(S[a], v, s), seas(S[b], v, s), v)
            draw(axes[r, c], d, f"{v} {s}", cmap="BrBG" if v == "pr" else "RdBu_r", sym=True,
                 sig=sig if s == "ANN" else None, label="%" if v == "pr" else "degC")
    fig.suptitle(lab, y=1.0)
    plt.tight_layout(); plt.savefig(OUT / f"signal_{a}_minus_{b}.png", dpi=110, bbox_inches="tight"); plt.show()

## 5. Zonal means over the domain (annual climatology)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 7))
for c, v in enumerate(("pr", "tasmax", "tasmin", "dtr")):
    for name, ds in S.items():
        z = seas(ds, v, "ANN").where(MASK).mean("lon")
        axes[0, c].plot(z.lat, z, color=COLORS[name], lw=1, label=RUNS[name])
    axes[0, c].set_title(f"zonal mean {v} ({UNITS[v]})")
    for a, b, lab, col in (("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original", "tab:red"),
                           ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original", "tab:orange")):
        if a in S and b in S:
            z = (seas(S[a], v, "ANN") - seas(S[b], v, "ANN")).where(MASK).mean("lon")
            axes[1, c].plot(z.lat, z, color=col, label=lab)
    axes[1, c].axhline(0, color="grey", lw=0.5); axes[1, c].set_title(f"hosing - original, zonal {v}")
    axes[1, c].set_xlabel("latitude")
axes[0, 0].legend(fontsize=7); axes[1, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "zonal_means.png", dpi=110); plt.show()

## 6. Annual cycle (means over the domain)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 7))
for c, v in enumerate(("pr", "tasmax", "tasmin", "dtr")):
    for name, ds in S.items():
        m = dmean(ds[f"{v}_clim_month"])
        axes[0, c].plot(m.month, m, "o-", ms=3, color=COLORS[name], label=RUNS[name])
    axes[0, c].set_title(f"annual cycle, {DOMAIN} {v} ({UNITS[v]})")
    for a, b, lab, col in (("gcm_2025_hosing", "gcm_2025_nohosing", "2025", "tab:red"),
                           ("gcm_2050_hosing", "gcm_2050_nohosing", "2050", "tab:orange")):
        if a in S and b in S:
            m = dmean(S[a][f"{v}_clim_month"]) - dmean(S[b][f"{v}_clim_month"])
            axes[1, c].plot(m.month, m, "s-", ms=3, color=col, label=f"{lab} hosing - original")
    axes[1, c].axhline(0, color="grey", lw=0.5); axes[1, c].set_xlabel("month")
axes[0, 0].legend(fontsize=7); axes[1, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "annual_cycle.png", dpi=110); plt.show()

## 7. Extremes: Rx1day (yearly max pr), TXx (yearly max tasmax), TNn (yearly min tasmin)

Top row: historical climatology of the yearly extremes. Other rows: hosing - original (Rx1day in %), dots = |Welch t| > 2.

In [ ]:
EXT = {"pr": "Rx1day", "tasmax": "TXx", "tasmin": "TNn"}
rows_ext = [("gcm_historical", None, "historical")] + [(a, b, f"{a[4:8]} hosing - original")
            for a, b in (("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing"))]
rows_ext = [r for r in rows_ext if r[0] in S and (r[1] is None or r[1] in S)]
if rows_ext:
    fig, axes = map_axes(len(rows_ext), 3)
    for r, (a, b, lab) in enumerate(rows_ext):
        for c, v in enumerate(VARS):
            x = S[a][f"{v}_year_ext"]
            if b is None:
                draw(axes[r, c], x.mean("year"), f"{lab} {EXT[v]}", cmap="YlGnBu" if v == "pr" else "RdYlBu_r",
                     lim=(0, float(np.nanpercentile(x.mean("year").where(MASK), 98))) if v == "pr" else None, label=UNITS[v])
            else:
                y = S[b][f"{v}_year_ext"]
                draw(axes[r, c], diff(x.mean("year"), y.mean("year"), v), f"{lab} {EXT[v]}",
                     cmap="BrBG" if v == "pr" else "RdBu_r", sym=True, sig=welch_sig(x, y),
                     label="%" if v == "pr" else "degC")
    plt.tight_layout(); plt.savefig(OUT / "extremes.png", dpi=110); plt.show()

## 8. Dry days and wet-day intensity

The WDF step of the bias adjustment acts on the dry-day frequency, and the pr QDM on wet-day intensities:
the hosing signal on both must be known. Dry-day differences in percentage points, intensity in %.

In [ ]:
rows_dry = [r for r in (("gcm_historical", None, "historical"),
                        ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
                        ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original"))
            if r[0] in S and (r[1] is None or r[1] in S)]
if rows_dry:
    fig, axes = map_axes(len(rows_dry), 2)
    for r, (a, b, lab) in enumerate(rows_dry):
        dry_a, int_a = S[a]["pr_year_dry"], S[a]["pr_year_sdii"]
        if b is None:
            draw(axes[r, 0], dry_a.mean("year"), f"{lab}: dry-day fraction", cmap="YlOrBr", lim=(0, 1))
            draw(axes[r, 1], int_a.mean("year"), f"{lab}: wet-day intensity", cmap="YlGnBu",
                 lim=(0, float(np.nanpercentile(int_a.mean("year").where(MASK), 98))), label="mm/day")
        else:
            dry_b, int_b = S[b]["pr_year_dry"], S[b]["pr_year_sdii"]
            draw(axes[r, 0], 100 * (dry_a.mean("year") - dry_b.mean("year")), f"{lab}: dry-day fraction",
                 cmap="BrBG_r", sym=True, sig=welch_sig(dry_a, dry_b), label="pp")
            draw(axes[r, 1], diff(int_a.mean("year"), int_b.mean("year"), "pr"), f"{lab}: wet-day intensity",
                 cmap="BrBG", sym=True, sig=welch_sig(int_a, int_b), label="%")
    plt.tight_layout(); plt.savefig(OUT / "dry_days_intensity.png", dpi=110); plt.show()

## 9. Pooled distributions (area-weighted sample of pixels of the domain, all days)

Quantiles of the daily values pooled over `N_SAMPLE` random pixels of the domain (the same pixels for every run).
Right panels: difference of each run to the historical run, quantile by quantile.

In [ ]:
qtab = pd.concat({RUNS[n]: pd.DataFrame({v: ds[f"{v}_q"].values for v in ("pr", "tasmax", "tasmin", "dtr")}
                                          | {"pr wet days": ds["pr_q_wet"].values}, index=QUANTILES)
                  for n, ds in S.items()}, axis=1)
qtab.index.name = "quantile"
qtab.to_csv(OUT / "pooled_quantiles.csv")
display(qtab.round(2))

fig, axes = plt.subplots(1, 4, figsize=(20, 3.8))
for ax, v in zip(axes, ("pr wet days", "tasmax", "tasmin", "dtr")):
    key = "pr_q_wet" if v == "pr wet days" else f"{v}_q"
    if "gcm_historical" not in S:
        break
    ref = S["gcm_historical"][key].values
    for name, ds in S.items():
        if name != "gcm_historical":
            ax.plot(QUANTILES, ds[key].values - ref, "o-", ms=3, color=COLORS[name], label=RUNS[name])
    ax.set_xscale("logit"); ax.axhline(0, color="grey", lw=0.5)
    ax.set_title(f"{v}: run - historical (quantile by quantile)"); ax.set_xlabel("quantile")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "pooled_quantile_differences.png", dpi=110); plt.show()

## 10. Interannual variability

Standard deviation of the yearly means, and its ratio hosing / original (1 = no change). A change of variability
is not corrected by a quantile mapping calibrated on the original run: good to know before interpreting results.

In [ ]:
rows_var = [r for r in (("gcm_2025_hosing", "gcm_2025_nohosing", "2025"), ("gcm_2050_hosing", "gcm_2050_nohosing", "2050"))
            if r[0] in S and r[1] in S]
if rows_var:
    fig, axes = map_axes(len(rows_var), 3)
    for r, (a, b, lab) in enumerate(rows_var):
        for c, v in enumerate(VARS):
            ratio = S[a][f"{v}_year_mean"].std("year") / S[b][f"{v}_year_mean"].std("year")
            draw(axes[r, c], ratio, f"{lab}: std(hosing) / std(original), yearly {v}", cmap="PuOr_r", lim=(0.5, 1.5))
    plt.tight_layout(); plt.savefig(OUT / "interannual_variability_ratio.png", dpi=110); plt.show()

## Things to note from this notebook (to fill in)

- Integrity: ...
- Stationarity of the 2025 / 2050 runs (trend per decade, second half - first half): ...
- Where the hosing signal is strong / significant (pr, tasmax, tasmin, extremes, dry days): ...
- Consequences for the bias adjustment (e.g. regions where the WDF or the QDM could interact with the hosing signal): ...

All figures and tables are saved in `<out_root>/exploration_gcm/`.